# Sistema de Análisis de Hojas de Vida
## NLP + BERT + Transformers

In [ ]:
# Importar módulos
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import os
from pathlib import Path
import fitz
import pytesseract
from PIL import Image
import io

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Función para extraer texto de PDF usando OCR
def extract_text_from_pdf(pdf_file):
    text = ""
    try:
        doc = fitz.open(pdf_file)
        for page_num, page in enumerate(doc):
            print(f"Procesando página {page_num + 1}...")
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
            img_data = pix.tobytes("png")
            image = Image.open(io.BytesIO(img_data))
            ocr_text = pytesseract.image_to_string(image, lang='eng')
            text += ocr_text or ""
        doc.close()
        print(f"Total extraído: {len(text)} caracteres")
    except Exception as e:
        print(f"Error: {e}")
    return text

## Cargar PDF para analizar

In [ ]:
# Cambia esta ruta al PDF que quieras analizar
PDF_PATH = "../../dataset/sample_cv.pdf"  # Ajusta la ruta

if os.path.exists(PDF_PATH):
    print(f"📄 Cargando PDF: {PDF_PATH}")
    pdf_text = extract_text_from_pdf(PDF_PATH)
    print(f"\nTexto extraído ({len(pdf_text)} caracteres):")
    print(pdf_text[:500] + "...")
else:
    print(f"⚠️ PDF no encontrado: {PDF_PATH}")
    pdf_text = None

## 1. Cargar Dataset

In [ ]:
from src.datasets.data_loader import DatasetLoader

loader = DatasetLoader()
datasets = loader.load_all()
print("Datasets cargados:", list(datasets.keys()))

## 2. Preparar Datos

In [ ]:
from src.datasets.preprocessor import DatasetPreprocessor

df = datasets['resume_class_training']
preprocessor = DatasetPreprocessor()
texts, labels = preprocessor.prepare_classification_data(df, "Resume Text", "Category")
train_texts, val_texts, train_labels, val_labels = preprocessor.prepare_for_bert(texts, labels, val_size=0.2)
label_mapping, reverse_mapping = preprocessor.get_label_mapping()
num_classes = len(label_mapping)
print(f"Train: {len(train_texts)}, Val: {len(val_texts)}")
print(f"Categorías: {num_classes}")

## 3. Cargar o Entrenar Modelo

In [ ]:
from src.models.bert_classifier import BertClassifierModel

MODEL_PATH = '../models/bert_classifier_category.pt'

models_dir = os.path.dirname(MODEL_PATH)
if models_dir and not os.path.exists(models_dir):
    os.makedirs(models_dir)
    print(f"📁 Carpeta '{models_dir}' creada.")

classifier = BertClassifierModel(num_classes=num_classes, device='cpu')

if os.path.exists(MODEL_PATH):
    print("✅ Cargando modelo guardado...")
    classifier.load_model(MODEL_PATH)
else:
    print("⚠️ Entrenando modelo por primera vez...")
    classifier.train(
        train_texts, train_labels,
        val_texts, val_labels,
        epochs=3,
        batch_size=16,
        save_path=MODEL_PATH
    )
    print("✅ Modelo guardado!")

## 4. Predicción con Porcentajes

### Analizar PDF cargado

In [ ]:
if pdf_text and len(pdf_text.strip()) > 10:
    from src.preprocessing.nlp_processor import get_nlp_processor
    from deep_translator import GoogleTranslator
    
    # Traducción
    print("Traduciendo texto...")
    try:
        translator = GoogleTranslator(source='auto', target='en')
        text_en = translator.translate(pdf_text)
    except:
        text_en = pdf_text
    
    # NLP Processing
    print("Extrayendo secciones...")
    nlp = get_nlp_processor()
    nlp_result = nlp.process(text_en)
    sections = nlp.extract_sections(text_en)
    
    # Experience + Skills
    experience_skills = ""
    if sections.get('experience'):
        experience_skills += sections['experience'] + " "
    if sections.get('skills'):
        experience_skills += sections['skills'] + " "
    if not experience_skills.strip():
        experience_skills = text_en
    
    # Limpiar texto
    from src.preprocessing.text_cleaner import TextCleaner
    cleaner = TextCleaner()
    cleaned_text = cleaner.clean(experience_skills)
    
    print(f"\nTexto para modelo: {len(cleaned_text)} caracteres")
    
    # Predicción
    predictions = predict_with_probabilities(classifier, cleaned_text, reverse_mapping)
    
    print("\n" + "=" * 60)
    print("PREDICCIÓN DE CATEGORÍA PROFESIONAL (DESDE PDF)")
    print("=" * 60)
    print(f"\n🎯 Categoría: {predictions[0]['category']}")
    print(f"📊 Confianza: {predictions[0]['probability']:.2f}%")
    print("\nTOP 10 CATEGORÍAS:")
    print("-" * 60)
    for i, pred in enumerate(predictions[:10]):
        bar = "█" * int(pred['probability'] / 2)
        print(f"{i+1:2}. {pred['category']:35} {pred['probability']:6.2f}% {bar}")
    
    # Mostrar entidades NER
    print("\n🏷️ Entidades detectadas:")
    if nlp_result['entities'].get('TECH'):
        print(f"  💻 Tecnologías: {nlp_result['entities']['TECH'][:10]}")
    if nlp_result['entities'].get('ORG'):
        print(f"  🏢 Organizaciones: {nlp_result['entities']['ORG'][:5]}")
    if nlp_result['entities'].get('PERSON'):
        print(f"  👤 Personas: {nlp_result['entities']['PERSON'][:5]}")
    
    # Skills from NER
    skills_ner = nlp.get_skills_from_ner(text_en)
    if skills_ner:
        print(f"\n🛠️ Skills detectados: {skills_ner[:15]}")
else:
    print("No hay texto del PDF para analizar")

### Ejemplo con texto manual

In [ ]:
sample_resume = """
EXPERIENCE
Software Engineer at Tech Corp
- Developed Python applications using Django and Flask
- Worked with PostgreSQL, MongoDB, and Redis
- Deployed applications using AWS and Docker

SKILLS
Python, JavaScript, Django, Flask, React, AWS, Docker, PostgreSQL, Git

EDUCATION
BS Computer Science - University of Technology
"""

print("CV:", sample_resume[:200], "...")

In [ ]:
from src.preprocessing.text_cleaner import TextCleaner
from src.preprocessing.nlp_processor import get_nlp_processor
import torch

# NLP Processing - Extract sections
print("Procesando NLP para extraer secciones...")
nlp = get_nlp_processor()
nlp_result = nlp.process(sample_resume)
sections = nlp.extract_sections(sample_resume)
print(f"Secciones encontradas: {list(sections.keys())}")

# Use Experience + Skills for prediction
experience_skills = ""
if sections.get('experience'):
    experience_skills += sections['experience'] + " "
if sections.get('skills'):
    experience_skills += sections['skills'] + " "
if not experience_skills.strip():
    experience_skills = sample_resume

print(f"Texto para modelo (Experience + Skills): {len(experience_skills)} caracteres")

cleaner = TextCleaner()
cleaned_text = cleaner.clean(experience_skills)
print("Texto limpio para BERT:", cleaned_text[:300], "...")

In [ ]:
# Predicción con probabilidades
def predict_with_probabilities(classifier, text, reverse_mapping, max_len=512):
    classifier.model.eval()
    encoding = classifier.tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )
    input_ids = encoding['input_ids'].to(classifier.device)
    attention_mask = encoding['attention_mask'].to(classifier.device)
    
    with torch.no_grad():
        outputs = classifier.model(input_ids, attention_mask)
        probs = torch.softmax(outputs, dim=1)
    
    probs = probs[0].cpu().numpy()
    
    results = []
    for idx, prob in enumerate(probs):
        category = reverse_mapping.get(idx, f"Unknown_{idx}")
        results.append({'category': category, 'probability': prob * 100})
    
    results.sort(key=lambda x: x['probability'], reverse=True)
    return results

predictions = predict_with_probabilities(classifier, cleaned_text, reverse_mapping)

print("=" * 60)
print("PREDICCIÓN DE CATEGORÍA PROFESIONAL")
print("=" * 60)
print(f"\n🎯 Categoría: {predictions[0]['category']}")
print(f"📊 Confianza: {predictions[0]['probability']:.2f}%")
print("\nTOP 10 CATEGORÍAS:")
print("-" * 60)
for i, pred in enumerate(predictions[:10]):
    bar = "█" * int(pred['probability'] / 2)
    print(f"{i+1:2}. {pred['category']:35} {pred['probability']:6.2f}% {bar}")

## 5. Extracción de Skills

In [ ]:
from src.preprocessing.skill_extractor import SkillExtractor

skills_list = loader.get_skills_list()
skill_extractor = SkillExtractor(skills_list)
extracted_skills = skill_extractor.extract_from_resume(sample_resume)

print("🛠️ Skills detectados:")
for skill in extracted_skills:
    print(f"  ✓ {skill}")

In [ ]:
print("\n" + "=" * 60)
print("RESUMEN FINAL")
print("=" * 60)
print(f"""
📄 CV ANALIZADO
   Caracteres: {len(cleaned_text)}

🎯 PREDICCIÓN
   Categoría: {predictions[0]['category']}
   Confianza: {predictions[0]['probability']:.2f}%
   Alternativas: {predictions[1]['category']} ({predictions[1]['probability']:.2f}%), {predictions[2]['category']} ({predictions[2]['probability']:.2f}%)

🛠️ SKILLS: {len(extracted_skills)} detectados

📊 Modelo: {num_classes} categorías
""")

---
## Notas:
- Si el modelo existe, lo carga automáticamente
- Si no existe, entrena y guarda
- El modelo se guarda en: ../models/bert_classifier_category.pt